In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder\
            .master('local[*]')\
            .appName('test_taxizones')\
            .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/07/23 02:34:08 WARN Utils: Your hostname, shawns-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.68.62 instead (on interface en0)
25/07/23 02:34:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/23 02:34:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/23 02:34:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv

--2025-07-23 02:35:27--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/5a2cc2f5-b4cd-4584-9c62-a6ea97ed0e6a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-07-22T19%3A29%3A29Z&rscd=attachment%3B+filename%3Dtaxi_zone_lookup.csv&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-07-22T18%3A29%3A28Z&ske=2025-07-22T19%3A29%3A29Z&sks=b&skv=2018-11-09&sig=aIF6oGR3Go8cUjH09GhXM%2FV73n1mZANoLkeoB0%2BkocA%3D&jwt=eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1MzIwOTYyNywibmJmIjoxNzUzMjA5MzI3LCJwYXRoIjoicmVsZWFzZWF

In [13]:
df = spark.read\
        .option('header', 'true')\
        .csv('taxi_zone_lookup.csv')

In [14]:
df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [17]:
print(df.schema)

StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])


In [19]:
df.write.parquet('zones')

In [20]:
import pyarrow.parquet as pq

# Load the Parquet file
parquet_file = pq.ParquetFile("./zones/part-00000-cda32122-7126-442c-a5e1-34a48ed57841-c000.snappy.parquet")

# Schema metadata (column types, structure)
print(parquet_file.schema)

# File-level metadata (key-value pairs, creator, etc.)
print(parquet_file.metadata)

# Specific metadata info
print(parquet_file.metadata.num_rows)       # Total number of rows
print(parquet_file.metadata.num_columns)    # Total number of columns
print(parquet_file.metadata.created_by)     # Creator info (e.g. 'pandas', 'spark')
print(parquet_file.metadata.row_group(0))   # Metadata for the first row group

required group field_id=-1 spark_schema {
  optional binary field_id=-1 LocationID (String);
  optional binary field_id=-1 Borough (String);
  optional binary field_id=-1 Zone (String);
  optional binary field_id=-1 service_zone (String);
}

  created_by: parquet-mr version 1.15.2 (build 859eac165b08f927fa14590c33bc5f476405fb68)
  num_columns: 4
  num_rows: 265
  num_row_groups: 1
  format_version: 1.0
  serialized_size: 997
265
4
parquet-mr version 1.15.2 (build 859eac165b08f927fa14590c33bc5f476405fb68)
  num_columns: 4
  num_rows: 265
  total_byte_size: 7492
  sorting_columns: ()
